In [1]:
### import modules ###
import os 
import pandas as pd
import xarray as xr
xr.set_options(display_expand_data=False)
import numpy as np

# Custom modules
from climate_indices_info_NOAA import NOAA_indices_info
from climate_indices_info_NCC import NCC_indices_info
from climate_indices_info_EXT import EXT_indices_info
indices_info = {}
indices_info.update(NOAA_indices_info)
indices_info.update(NCC_indices_info)
indices_info.update(EXT_indices_info)
print(len(indices_info))

259


In [2]:
def climate_indice_load(indices_name):
    climate_indices_dir = "/home/huangwenshuo/Program/RCSD/data/climate_indices"
    path_list = []
    # 递归获得文件夹和子文件夹下所有文件名
    for root,dirs,files in os.walk(climate_indices_dir):
        for file in files:
            path = os.path.join(root,file)
            # print(path)
            path_list.append(path)
    def find_file_path_by_indices(indices_name, file_list):
        # 创建一个匹配的文件名
        target_file_name = f"{indices_name}.txt"  # 根据你的文件命名规则调整
        for file_path in file_list:
            if target_file_name in file_path:
                return file_path
        return None  # 如果没有找到返回 None
    file_name = indices_info["%s"%indices_name]['file_name']
    full_path = find_file_path_by_indices(file_name, path_list)
    #read time coverage
    years = pd.read_csv(full_path, nrows=1, sep='\s+')
    start_year = int(years.columns[0])
    end_year = int(years.columns[-1])
    temporal_coverage = "%s-%s"%(start_year,end_year)
    print("Time coverage: %s-%s"%(start_year,end_year))
    #read data value 
    ds = pd.read_csv(full_path,header=None,sep='\s+', skiprows=1,index_col=False,on_bad_lines='skip')
    # ds.set_index(0,inplace=True)
    ds = ds.loc[0:end_year-start_year].iloc[:,1:13].astype(float)
    ds
    #flatten the data
    years_range = range(start_year, end_year+1)
    CItime = xr.cftime_range(start="%s-01-01"%start_year,freq="MS",periods=(end_year+1-start_year)*12)
    climate_indice = xr.DataArray(ds.values.flatten(), dims="time", coords={"time": CItime})
    climate_indice = climate_indice.rename(indices_info["%s"%indices_name]['short_name'])
    return climate_indice, temporal_coverage

In [3]:
# check indices information 
indices_info

{'Niño 1+2 (HadISST) ': {'standard_name': 'Niño 1+2 (HadISST) ',
  'short_name': 'Niño 1+2 (HadISST) ',
  'temporal_coverage': '1870-2024',
  'temporal_resolution': 'monthly',
  'file_name': 'nino12.long.anom.data',
  'update_link': 'https://psl.noaa.gov/data/timeseries/month/data/nino12.long.anom.data',
  'description': 'Extreme Eastern Tropical Pacific SST anomaly 0N-10S, 90W-80W',
  'web_link': 'https://psl.noaa.gov/data/timeseries/month/DS/Nino12/',
  'Source': 'NOAA/PSL'},
 'Niño 1+2 (ERSSTv5) ': {'standard_name': 'Niño 1+2 (ERSSTv5) ',
  'short_name': 'Niño 1+2 (ERSSTv5) ',
  'temporal_coverage': '1948-2024',
  'temporal_resolution': 'monthly',
  'file_name': 'nina1.anom.data',
  'update_link': 'https://www.psl.noaa.gov/data/correlation/nina1.anom.data',
  'description': 'Extreme Eastern Tropical Pacific SST anomaly 0N-10S, 90W-80W',
  'web_link': 'https://psl.noaa.gov/data/timeseries/month/DS/Nino12_CPC/',
  'Source': 'NOAA/CPC'},
 'Niño 3 (HadISST)': {'standard_name': 'Niño 3 (

In [3]:
#create climate indices list for README.md
for index, info in indices_info.items():
    print('-'+'[%s]'%index+"(%s)"%indices_info[index]['web_link']+" (%s)"%indices_info[index]['temporal_coverage']+", ")
    print(indices_info[index]['standard_name']+"  ") #末尾换行

-[Niño 1+2 (HadISST) ](https://psl.noaa.gov/data/timeseries/month/DS/Nino12/) (1870-2024), 
Niño 1+2 (HadISST)   
-[Niño 1+2 (ERSSTv5) ](https://psl.noaa.gov/data/timeseries/month/DS/Nino12_CPC/) (1948-2024), 
Niño 1+2 (ERSSTv5)   
-[Niño 3 (HadISST)](https://psl.noaa.gov/data/timeseries/month/DS/Nino3/) (1870-2024), 
Niño 3 (HadISST)  
-[Niño 3 (ERSSTv5)](https://psl.noaa.gov/data/timeseries/month/DS/Nino3_CPC/) (1948-2024), 
Niño 3 (ERSSTv5)  
-[Niño 3.4 (HadISST)](https://psl.noaa.gov/data/timeseries/month/DS/Nino34/) (1870-2024), 
Niño 3.4 (HadISST)  
-[Niño 3.4 (ERSSTv5)](https://psl.noaa.gov/data/timeseries/month/DS/Nino34_CPC/) (1948-2024), 
Niño 3.4 (ERSSTv5)  
-[Niño 4 (HadISST)](https://psl.noaa.gov/data/timeseries/month/DS/Nino4/) (1870-2024), 
Niño 4 (HadISST)  
-[Niño 4 (ERSSTv5)](https://psl.noaa.gov/data/timeseries/month/DS/Nino4_CPC/) (1948-2024), 
Niño 4 (ERSSTv5)  
-[MEI_Extended](https://psl.noaa.gov/data/timeseries/month/DS/MEI_EXTENDED/) (1871-2005), 
Multivariate 

In [ ]:
# debug standname#
# for index, info in indices_info.items():
#     print(f"{index}")
# climate_indice_load function test #
for value in indices_info.values():
    print("Now is testing " + value['short_name'])
    _, tc = climate_indice_load(value['short_name'])
    indices_info[value['short_name']]['temporal_coverage'] = tc

In [5]:
# debug temporal coverage#
# for index, info in indices_info.items():
#     print(f"{index}")
# climate_indice_load function test #
for value in indices_info.values():
    print("Now is testing " + value['short_name'])
    _, tc = climate_indice_load(value['short_name'])
    indices_info[value['short_name']]['temporal_coverage'] = tc

Now is testing Niño 1+2 (HadISST) 
Time coverage: 1870-2024
Now is testing Niño 1+2 (ERSSTv5) 
Time coverage: 1948-2024
Now is testing Niño 3 (HadISST)
Time coverage: 1870-2024
Now is testing Niño 3 (ERSSTv5)
Time coverage: 1948-2024
Now is testing Niño 3.4 (HadISST)
Time coverage: 1870-2024
Now is testing Niño 3.4 (ERSSTv5)
Time coverage: 1948-2024
Now is testing Niño 4 (HadISST)
Time coverage: 1870-2024
Now is testing Niño 4 (ERSSTv5)
Time coverage: 1948-2024
Now is testing MEI_Extended
Time coverage: 1871-2005
Now is testing MEI_V2
Time coverage: 1979-2024
Now is testing ONI
Time coverage: 1950-2024
Now is testing OLR_CTP
Time coverage: 1974-2024
Now is testing WH_WP
Time coverage: 1948-2024
Now is testing PAC_WP
Time coverage: 1948-2024
Now is testing PAC_WP_Long
Time coverage: 1854-2024
Now is testing TNI
Time coverage: 1948-2023
Now is testing TNI_Long
Time coverage: 1870-2023
Now is testing HeatCentraPac
Time coverage: 1979-2024
Now is testing BEST
Time coverage: 1948-2023
Now i

In [8]:
#create climate indices list for README.md
for index, info in indices_info.items():
    print('[%s'%index)
    # print(indices_info[index]['standard_name']+"  ") #末尾换行

[Niño 1+2 (HadISST) 
[Niño 1+2 (ERSSTv5) 
[Niño 3 (HadISST)
[Niño 3 (ERSSTv5)
[Niño 3.4 (HadISST)
[Niño 3.4 (ERSSTv5)
[Niño 4 (HadISST)
[Niño 4 (ERSSTv5)
[MEI_Extended
[MEI_V2
[ONI
[OLR_CTP
[WH_WP
[PAC_WP
[PAC_WP_Long
[TNI
[TNI_Long
[HeatCentraPac
[BEST_long
[BEST
[AMM_WIND
[AMM_SST
[PMM_WIND
[PMM_SST
[DMI/IOD
[DMI/IOD EAST
[TC-NE Pacific Days
[TC-Atlantic Days
[TC-NW Pacific Days
[TC-NE Pacific ACE
[TC-Atlantic ACE
[TC-NW Pacific ACE
[NPGO
[PDO_ENS
[PDO_ERSSTv5
[PDO_HadISST_1.1
[PDO_COBE2
[PDO_CPC
[TPI_HadISST_2
[TPI_ERSSTv5
[TPI_HadISST_1.1
[TPI_COBE
[Tropical Pacific SST EOF
[Atlantic Tripole SST EOF
[AMO_us
[AMO_sm
[AMO_us_long
[AMO_sm_long
[GSL
[AAO
[AO
[EA/WR
[AO_20CRv3
[Madras-SLP
[NJ-SLP
[NAO_AZO
[NAO
[NAO_ICE
[NAO_GIB
[NAO_Jones
[NAO_CRU_Long
[RNAO
[EP/NP
[NOI
[NP
[NP/NPI
[NP_20CRV3
[TNA
[TSA
[NTA
[CAR
[PNA
[SCAND
[SOI_CRU_1866
[SOI
[SOI_20CRV3
[WP
[CO2-Loa
[CO2-GlobalAve
[GBI_long
[GBI
[GLAAM
[MDR
[QBO-30mb
[QBO-50mb
[uwnd_200_troppac
[Brazil_Pr
[ENSO_Pr
[Sahel_Pr_Long
[Sahel

In [ ]:
/Niño 1+2 (HadISST) /Niño 1+2 (ERSSTv5) /Niño 3 (HadISST)/Niño 3 (ERSSTv5)/Niño 3.4 (HadISST)/Niño 3.4 (ERSSTv5)/Niño 4 (HadISST)/Niño 4 (ERSSTv5)/MEI_Extended/MEI_V2/ONI/OLR_CTP/WH_WP/PAC_WP/PAC_WP_Long/TNI/TNI_Long/HeatCentraPac/BEST_long/BEST/AMM_WIND/AMM_SST/PMM_WIND/PMM_SST/DMI/IOD/DMI/IOD EAST/TC-NE Pacific Days/TC-Atlantic Days/TC-NW Pacific Days/TC-NE Pacific ACE/TC-Atlantic ACE/TC-NW Pacific ACE/NPGO/PDO_ENS/PDO_ERSSTv5/PDO_HadISST_1.1/PDO_COBE2/PDO_CPC/TPI_HadISST_2/TPI_ERSSTv5/TPI_HadISST_1.1/TPI_COBE/Tropical Pacific SST EOF/Atlantic Tripole SST EOF/AMO_us/AMO_sm/AMO_us_long/AMO_sm_long/GSL/AAO/AO/EA/WR/AO_20CRv3/Madras-SLP/NJ-SLP/NAO_AZO/NAO/NAO_ICE/NAO_GIB/NAO_Jones/NAO_CRU_Long/RNAO/EP/NP/NOI/NP/NP/NPI/NP_20CRV3/TNA/TSA/NTA/CAR/PNA/SCAND/SOI_CRU_1866/SOI/SOI_20CRV3/WP/CO2-Loa/CO2-GlobalAve/GBI_long/GBI/GLAAM/MDR/QBO-30mb/QBO-50mb/uwnd_200_troppac/Brazil_Pr/ENSO_Pr/Sahel_Pr_Long/Sahel_Pr/NSierra8_Pr/NSierra5_Pr/NSierra6_Pr/SWM_Pr/SWUSM_Pr_Long/CI_Pr/UCB_Pr/NASA_Land_T/NASA_Comb_T/Berk_Land_T/Berk_Comb_T/CRU_Land_T/CRU_Comb_T/NOAA_Land_T/NOAA_Comb_T/NH_Ice_Area/NH_Ice_Extent/NH_Snow_Extent/SH_Ice_Area/SH_Ice_Extent/Sunspot_Count/Solar Flux/US_Tornado_Count/UCB_T/LeesFerryNatlFlow/NCC_ATM1/NCC_ATM2/NCC_ATM3/NCC_ATM4/NCC_ATM5/NCC_ATM6/NCC_ATM7/NCC_ATM8/NCC_ATM9/NCC_ATM10/NCC_ATM11/NCC_ATM12/NCC_ATM13/NCC_ATM14/NCC_ATM15/NCC_ATM16/NCC_ATM17/NCC_ATM18/NCC_ATM19/NCC_ATM20/NCC_ATM21/NCC_ATM22/NCC_ATM23/NCC_ATM24/NCC_ATM25/NCC_ATM26/NCC_ATM27/NCC_ATM28/NCC_ATM29/NCC_ATM30/NCC_ATM31/NCC_ATM32/NCC_ATM33/NCC_ATM34/NCC_ATM35/NCC_ATM36/NCC_ATM38/NCC_ATM39/NCC_ATM40/NCC_ATM41/NCC_ATM43/NCC_ATM44/NCC_ATM45/NCC_ATM46/NCC_ATM47/NCC_ATM48/NCC_ATM49/NCC_ATM50/NCC_ATM51/NCC_ATM52/NCC_ATM53/NCC_ATM54/NCC_ATM55/NCC_ATM56/NCC_ATM57/NCC_ATM58/NCC_ATM59/NCC_ATM60/NCC_ATM61/NCC_ATM62/NCC_ATM63/NCC_ATM64/NCC_ATM65/NCC_ATM66/NCC_ATM67/NCC_ATM68/NCC_ATM69/NCC_ATM70/NCC_ATM71/NCC_ATM72/NCC_ATM73/NCC_ATM74/NCC_ATM75/NCC_ATM76/NCC_ATM77/NCC_ATM78/NCC_ATM80/NCC_ATM81/NCC_ATM82/NCC_ATM83/NCC_ATM84/NCC_ATM85/NCC_ATM86/NCC_ATM87/NCC_ATM88/NCC_OCE1/NCC_OCE2/NCC_OCE3/NCC_OCE4/NCC_OCE5/NCC_OCE6/NCC_OCE7/NCC_OCE8/NCC_OCE9/NCC_OCE10/NCC_OCE11/NCC_OCE12/NCC_OCE13/NCC_OCE14/NCC_OCE15/NCC_OCE16/NCC_OCE17/NCC_OCE18/NCC_OCE19/NCC_OCE20/NCC_OCE21/NCC_OCE22/NCC_OCE23/NCC_OCE24/NCC_OCE25/NCC_OCE26/NCC_EXT2/NCC_EXT3/NCC_EXT4/NCC_EXT5/NCC_EXT6/NCC_EXT7/NCC_EXT9/NCC_EXT10/NCC_EXT11/NCC_EXT12/NCC_EXT13/NCC_EXT14/NCC_EXT15/NCC_EXT16/amoc_mean/amoc_std/amoc_cglo/amoc_glos/amoc_glor/amoc_oras/SAM_Station/SAM_20CRV3/SAM_20crv2c/PJ_huang04/PJ_noh21/PJ_ling22/SINTEX_ATL3/SINTEX_CNI/SINTEX_DNI/SINTEX_NNI/SINTEX_SASD/SINTEX_SIOD/IOBM_IRCSD
